# MVA v3 · NB04C3 — L737* splice-junction directionality and exon-skip audit
**v1.0.1 · public-variant-only · AlphaGenome research use only**

## Trigger

NB04C2 isolated one BUB1B junction:

- `chr15:40208770–40210109`
- 92 extreme AlphaGenome tracks
- max absolute quantile = 0.99615246

The recommended AlphaGenome `SpliceJunctionScorer` is **unsigned**: its headline score is based on the maximum
absolute REF↔ALT change. NB04C2 therefore cannot tell whether the L737* ALT allele **increases** or **decreases**
this junction.

## Biological question

Does `chr15:40209701 T>G` (BUB1B L737*) increase a junction that skips the coding interval containing the
premature stop codon?

## Frozen direction rule before viewing raw REF/ALT predictions

The previously frozen NB04C2 extreme-track set is reused unchanged.

For matched extreme tracks:

- `ALT_INCREASES_TARGET_JUNCTION` if >=80% have `ALT > REF`;
- `ALT_DECREASES_TARGET_JUNCTION` if >=80% have `ALT < REF`;
- otherwise `DIRECTION_HETEROGENEOUS`.

At least 20 previously extreme tracks must be matched. This rule is descriptive/model-internal only and is not
clinical evidence.

## Transcript-structure audit

Public Ensembl CDS mapping is queried for MANE transcript `ENST00000287598` at:
`c.2143`, `c.2144`, `c.2284`, `c.2285`.

The exon-skip interpretation is supported only if:

- `c.2143` maps to junction start;
- `c.2285` maps to junction end;
- L737* `c.2210` lies inside the skipped CDS interval `c.2144–c.2284`.

The skipped coding interval is 141 nt. If the mapping is confirmed, a frame-preserving transcript is
**plausible** because 141 is divisible by three; the actual junction transcript/protein must still be measured.

## v1.0.1 technical fix

The current AlphaGenome `DnaClient.predict_variant()` signature requires the keyword-only
`ontology_terms` argument. `ontology_terms=None` explicitly requests all available ontologies/tracks,
which is exactly the intended NB04C3 behavior because directionality is evaluated across the frozen
NB04C2 extreme-track set.

No preregistered biological rule, threshold, target junction, or comparator set is changed.


In [ ]:
import os
import importlib.util, subprocess, sys, os, json, math, requests, re
for mod,pkg in [("alphagenome","alphagenome"),("pandas","pandas"),("numpy","numpy")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","-U",pkg])

import numpy as np
import pandas as pd
from pathlib import Path
from alphagenome.data import genome
from alphagenome.models import dna_client, dna_output

API_KEY=None
try:
    from google.colab import userdata
    API_KEY=userdata.get("ALPHA_GENOME_API_KEY")
except Exception:
    API_KEY=os.environ.get("ALPHA_GENOME_API_KEY")
assert API_KEY, "Set ALPHA_GENOME_API_KEY in Colab Secrets/environment."

model=dna_client.create(
    api_key=API_KEY,
    address="dns:///gdmscience.googleapis.com:443",
)

from google.colab import drive
drive.mount("/content/drive",force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
PUBLIC_OUT=ROOT/"03_PUBLIC_ARTIFACTS"
DOC_OUT=ROOT/"04_DOCUMENTATION"
PUBLIC_OUT.mkdir(parents=True,exist_ok=True)
DOC_OUT.mkdir(parents=True,exist_ok=True)

C2_TRACKS=PUBLIC_OUT/"NB04C2_L737STAR_TARGET_JUNCTION_TRACKS.csv"
assert C2_TRACKS.exists(), f"Missing prerequisite: {C2_TRACKS}"

VARIANT=genome.Variant(
    chromosome="chr15",
    position=40209701,
    reference_bases="T",
    alternate_bases="G",
)
TARGET_START=40208770
TARGET_END=40210109
ENST="ENST00000287598"

print("Public variant:",VARIANT)

## 1. Verify MANE CDS coordinates independently with Ensembl REST

In [ ]:
def ensembl_map_cds(cds_pos):
    url=f"https://rest.ensembl.org/map/cds/{ENST}/{cds_pos}..{cds_pos}"
    r=requests.get(
        url,
        headers={"Content-Type":"application/json","Accept":"application/json"},
        timeout=90,
    )
    r.raise_for_status()
    data=r.json()
    maps=data.get("mappings",[])
    assert maps, f"No mapping for CDS c.{cds_pos}"
    # Single-base CDS query should yield one genomic mapping.
    m=maps[0]
    return {
        "cds_pos":int(cds_pos),
        "seq_region_name":str(m["seq_region_name"]),
        "genomic_start":int(m["start"]),
        "genomic_end":int(m["end"]),
        "strand":int(m["strand"]),
        "assembly_name":m.get("assembly_name"),
        "coordinate_system":m.get("coordinate_system"),
    }

cds_map=pd.DataFrame([ensembl_map_cds(x) for x in [2143,2144,2210,2284,2285]])
display(cds_map)

cds_map.to_csv(PUBLIC_OUT/"NB04C3_BUB1B_CDS_MAPPING.csv",index=False)

def gpos(c):
    r=cds_map[cds_map.cds_pos==c].iloc[0]
    assert r.genomic_start==r.genomic_end
    return int(r.genomic_start)

mapping_checks={
    "c2143_equals_junction_start": gpos(2143)==TARGET_START,
    "c2285_equals_junction_end": gpos(2285)==TARGET_END,
    "L737star_c2210_inside_skipped_interval": 2144 <= 2210 <= 2284,
    "skipped_coding_nt": 2284-2144+1,
    "skipped_nt_divisible_by_3": ((2284-2144+1)%3)==0,
}
print(json.dumps(mapping_checks,indent=2))

## 2. Obtain raw REF and ALT splice-junction predictions

In [ ]:
interval=VARIANT.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

vo=model.predict_variant(
    interval=interval,
    variant=VARIANT,
    ontology_terms=None,  # required by current client; None = return all ontologies/tracks
    requested_outputs=[dna_output.OutputType.SPLICE_JUNCTIONS],
)

ref=vo.reference.splice_junctions
alt=vo.alternate.splice_junctions
assert ref is not None and alt is not None
assert ref.values.shape[1]==alt.values.shape[1]
assert list(ref.metadata["name"])==list(alt.metadata["name"])

def junction_key(j):
    return (str(j.chromosome),int(j.start),int(j.end),str(j.strand))

ref_keys=[junction_key(j) for j in ref.junctions]
alt_keys=[junction_key(j) for j in alt.junctions]

# AlphaGenome JunctionData uses genomic Interval/Junction coordinates.
# Match exact NB04C2 coordinates first; tolerate a +/-1 representation difference only as an explicit warning.
def target_indices(keys):
    exact=[i for i,k in enumerate(keys) if k[1]==TARGET_START and k[2]==TARGET_END]
    if exact:
        return exact,0
    for off in [-1,1]:
        hit=[i for i,k in enumerate(keys) if k[1]==TARGET_START+off and k[2]==TARGET_END+off]
        if hit:
            return hit,off
    return [],None

ri,roff=target_indices(ref_keys)
ai,aoff=target_indices(alt_keys)
assert ri and ai, (
    f"Target junction not found in raw predictions. "
    f"REF nearby={sorted(ref_keys,key=lambda k:abs(k[1]-TARGET_START)+abs(k[2]-TARGET_END))[:5]}, "
    f"ALT nearby={sorted(alt_keys,key=lambda k:abs(k[1]-TARGET_START)+abs(k[2]-TARGET_END))[:5]}"
)
assert roff==aoff
assert len(ri)==1 and len(ai)==1

ridx=ri[0]; aidx=ai[0]
assert ref_keys[ridx]==alt_keys[aidx], (ref_keys[ridx],alt_keys[aidx])
print("Matched raw target junction:",ref_keys[ridx],"coordinate_offset:",roff)

## 3. Compare ALT vs REF per AlphaGenome RNA-seq track

In [ ]:
meta=ref.metadata.reset_index(drop=True).copy()
meta["ref_pred"]=np.asarray(ref.values[ridx,:],dtype=float)
meta["alt_pred"]=np.asarray(alt.values[aidx,:],dtype=float)
meta["alt_minus_ref"]=meta["alt_pred"]-meta["ref_pred"]
meta["direction"]=np.where(
    meta.alt_minus_ref>0,"ALT_GT_REF",
    np.where(meta.alt_minus_ref<0,"ALT_LT_REF","EQUAL")
)

# Log2 ratio is descriptive only; direction itself comes directly from raw ALT vs REF predictions.
EPS=1e-8
meta["log2_alt_over_ref"]=np.log2((meta.alt_pred+EPS)/(meta.ref_pred+EPS))

# Normalize track labels only for joining to the already-frozen NB04C2 track set.
def norm_name(x):
    s=str(x)
    s=re.sub(r"^junction_","",s)
    return re.sub(r"\s+"," ",s).strip()

meta["track_norm"]=meta["name"].map(norm_name)

c2=pd.read_csv(C2_TRACKS)
c2_ext=c2[c2["extreme"].astype(str).str.lower().eq("true")].copy()
c2_ext["track_norm"]=c2_ext["track_name"].map(norm_name)

extreme_names=set(c2_ext.track_norm)
meta["was_NB04C2_extreme"]=meta.track_norm.isin(extreme_names)

matched=meta[meta.was_NB04C2_extreme].copy()
assert len(matched)>=20, (
    f"Only {len(matched)} NB04C2 extreme tracks matched raw predictions; "
    "do not issue a direction call."
)

print("Previously extreme tracks:",len(c2_ext))
print("Matched to raw predictions:",len(matched))
display(
    matched[
        ["name","ref_pred","alt_pred","alt_minus_ref","log2_alt_over_ref","direction"]
    ].sort_values("alt_minus_ref",ascending=False).head(40)
)

meta.to_csv(PUBLIC_OUT/"NB04C3_L737STAR_RAW_REF_ALT_ALL_TRACKS.csv",index=False)
matched.to_csv(PUBLIC_OUT/"NB04C3_L737STAR_RAW_REF_ALT_EXTREME_TRACKS.csv",index=False)

## 4. Frozen direction call

In [ ]:
informative=matched[matched.direction!="EQUAL"].copy()
assert len(informative)>=20

frac_up=float((informative.direction=="ALT_GT_REF").mean())
frac_down=float((informative.direction=="ALT_LT_REF").mean())

if frac_up>=0.80:
    direction_call="ALT_INCREASES_TARGET_JUNCTION"
elif frac_down>=0.80:
    direction_call="ALT_DECREASES_TARGET_JUNCTION"
else:
    direction_call="DIRECTION_HETEROGENEOUS"

summary={
    "n_NB04C2_extreme_tracks":int(len(c2_ext)),
    "n_extreme_tracks_matched":int(len(matched)),
    "n_informative_tracks":int(len(informative)),
    "fraction_ALT_gt_REF":frac_up,
    "fraction_ALT_lt_REF":frac_down,
    "median_REF_prediction":float(matched.ref_pred.median()),
    "median_ALT_prediction":float(matched.alt_pred.median()),
    "median_ALT_minus_REF":float(matched.alt_minus_ref.median()),
    "median_log2_ALT_over_REF":float(matched.log2_alt_over_ref.replace([np.inf,-np.inf],np.nan).median()),
    "direction_call":direction_call,
}
print(json.dumps(summary,indent=2))

## 5. Integrate direction with transcript structure

In [ ]:
structure_confirmed=all([
    mapping_checks["c2143_equals_junction_start"],
    mapping_checks["c2285_equals_junction_end"],
    mapping_checks["L737star_c2210_inside_skipped_interval"],
])

if not structure_confirmed:
    biological_call="TRANSCRIPT_MAPPING_MISMATCH_HOLD"
elif direction_call=="ALT_INCREASES_TARGET_JUNCTION":
    biological_call="MODEL_SUPPORTS_INCREASED_FRAME_PRESERVING_EXON_SKIP"
elif direction_call=="ALT_DECREASES_TARGET_JUNCTION":
    biological_call="MODEL_SUPPORTS_DECREASED_EXON_SKIP_JUNCTION"
else:
    biological_call="EXON_SKIP_JUNCTION_DIRECTION_HETEROGENEOUS"

print("Structure confirmed:",structure_confirmed)
print("Biological model call:",biological_call)
print("Skipped CDS bases:",mapping_checks["skipped_coding_nt"])
print("Frame-preserving plausible:",mapping_checks["skipped_nt_divisible_by_3"])

## 6. Sanitized handoff

In [ ]:
from datetime import datetime, timezone

handoff={
    "schema":"MVA-NB04C3-L737STAR-DIRECTIONALITY-1.0.1",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "status":"COMPLETE",
    "variant":"BUB1B L737* / chr15:40209701:T>G",
    "target_junction":{
        "start":TARGET_START,
        "end":TARGET_END,
        "raw_prediction_coordinate_offset":roff,
    },
    "transcript":"ENST00000287598 / NM_001211.6",
    "transcript_mapping":{
        "cds_positions":cds_map.to_dict(orient="records"),
        **mapping_checks,
    },
    "direction_summary":summary,
    "biological_model_call":biological_call,
    "interpretation":{
        "if_increased":(
            "AlphaGenome predicts higher ALT than REF counts for the junction that skips CDS c.2144-c.2284, "
            "which contains L737*. Because the skipped interval is 141 nt, a frame-preserving exon-skip transcript "
            "is plausible. This must be validated by junction-specific RT-PCR/cDNA sequencing."
        ),
        "if_decreased":(
            "AlphaGenome predicts lower ALT than REF counts for the exon-skipping junction; the L737* allele does "
            "not support a compensatory skipping interpretation from this model."
        ),
        "if_heterogeneous":(
            "Raw AlphaGenome predictions do not support a consistent direction across previously extreme tracks."
        ),
    },
    "experimental_update":{
        "required_assay":"junction-specific RT-PCR followed by cDNA sequencing",
        "additional_readouts":[
            "canonical exon-inclusion transcript",
            "target exon-skip transcript",
            "total BUB1B RNA",
            "allele-aware transcript abundance if feasible",
            "BUBR1 protein abundance"
        ],
        "reason":"Model directionality can prioritize the assay but cannot establish the transcript in patient/isogenic RNA."
    },
    "acmg_rule":{
        "alphagenome_is_PS3":False,
        "alphagenome_is_additional_PP3":False,
        "clinical_use":False,
    },
    "privacy":{
        "private_VCF_used":False,
        "private_patient_genotypes_sent":False,
        "only_already_public_L737star_variant_sent":True,
    },
    "terms_notice":"Research-only AlphaGenome-derived output; subject to AlphaGenome Output Terms of Use and not for clinical decision-making."
}

(DOC_OUT/"NB04C3_L737STAR_DIRECTIONALITY_HANDOFF.json").write_text(
    json.dumps(handoff,indent=2)
)
print(json.dumps(handoff,indent=2))